In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/m5_sales_raw.csv")

In [5]:
df.head()

,date,sku_id,category,subcategory,price,units_sold,revenue,promotion_flag,stock_available
0,2023-12-18,SKU001,Electronics,Accessories,155.30,19,2950.62,False,8
1,2023-12-18,SKU002,Electronics,Audio,123.64,0,0.00,False,116
2,2023-12-18,SKU003,Electronics,Accessories,27.98,25,699.48,False,63
3,2023-12-18,SKU004,Electronics,Phones,130.11,100,13010.82,False,10
4,2023-12-18,SKU005,Electronics,Laptops,19.76,71,1403.27,False,87


In [6]:
df = df[["date", "sku_id", "category", "units_sold"]]
df.head()
df.info()
df.describe()
df.isnull().sum()
df.duplicated().sum()
df.drop_duplicates(inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365000 entries, 0 to 364999
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        365000 non-null  object
 1   sku_id      365000 non-null  object
 2   category    365000 non-null  object
 3   units_sold  365000 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 11.1+ MB


In [7]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["sku_id", "date"])

In [8]:
df.head()

,date,sku_id,category,units_sold
0,2023-12-18,SKU001,Electronics,19
500,2023-12-19,SKU001,Electronics,26
1000,2023-12-20,SKU001,Electronics,35
1500,2023-12-21,SKU001,Electronics,15
2000,2023-12-22,SKU001,Electronics,26


In [11]:
full_df = (
    df.set_index("date")
    .groupby("sku_id", group_keys=False)
    .apply(lambda x: x.asfreq("D", fill_value=0))
    .reset_index()
)


C:\Users\aloag\AppData\Local\Temp\ipykernel_22600\3309816971.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.asfreq("D", fill_value=0))


In [12]:
full_df.head()

,date,sku_id,category,units_sold
0,2023-12-18,SKU001,Electronics,19
1,2023-12-19,SKU001,Electronics,26
2,2023-12-20,SKU001,Electronics,35
3,2023-12-21,SKU001,Electronics,15
4,2023-12-22,SKU001,Electronics,26


In [13]:
# lag features
for lag in [7, 14, 28]:
    full_df[f"lag_{lag}"] = (
        full_df.groupby("sku_id")["units_sold"].shift(lag)
    )


In [14]:
# rolling features
for win in [7, 14, 28]:
    full_df[f"roll_mean_{win}"] = (
        full_df.groupby("sku_id")["units_sold"]
        .shift(1)
        .rolling(win)
        .mean()
    )

In [15]:
# calendar features
full_df["day_of_week"] = full_df["date"].dt.weekday
full_df["week_of_year"] = full_df["date"].dt.isocalendar().week.astype(int)
full_df["month"] = full_df["date"].dt.month


In [16]:
# Encode category
full_df["category_code"] = full_df["category"].astype("category").cat.codes


In [17]:
full_df = full_df.dropna()

In [18]:
# Time based split
split_date = full_df["date"].max() - pd.Timedelta(days=28)

train_df = full_df[full_df["date"] < split_date]
valid_df = full_df[full_df["date"] >= split_date]


In [19]:
features = [
    "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_mean_14", "roll_mean_28",
    "day_of_week", "week_of_year", "month",
    "category_code"
]

target = "units_sold"


In [20]:
train_df[features + [target]].to_csv("../data/processed/train_processed.csv", index=False)
valid_df[features + [target]].to_csv("../data/processed/valid_processed.csv", index=False)


In [21]:
train_df

,date,sku_id,category,units_sold,lag_7,lag_14,lag_28,roll_mean_7,roll_mean_14,roll_mean_28,day_of_week,week_of_year,month,category_code
28,2024-01-15,SKU001,Electronics,25,13.0,12.0,19.0,12.571429,13.000000,17.535714,0,3,1,2
29,2024-01-16,SKU001,Electronics,13,6.0,13.0,26.0,14.285714,13.928571,17.750000,1,3,1,2
30,2024-01-17,SKU001,Electronics,23,17.0,7.0,35.0,15.285714,13.928571,17.285714,2,3,1,2
31,2024-01-18,SKU001,Electronics,14,17.0,13.0,15.0,16.142857,15.071429,16.857143,3,3,1,2
32,2024-01-19,SKU001,Electronics,12,9.0,16.0,26.0,15.714286,15.142857,16.821429,4,3,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
364966,2025-11-13,SKU500,Books,56,35.0,0.0,40.0,52.571429,45.071429,42.142857,3,46,11,0
364967,2025-11-14,SKU500,Books,35,42.0,47.0,34.0,55.571429,49.071429,42.714286,4,46,11,0
364968,2025-11-15,SKU500,Books,41,81.0,0.0,55.0,54.571429,48.214286,42.750000,5,46,11,0
364969,2025-11-16,SKU500,Books,65,0.0,103.0,33.0,48.857143,51.142857,42.250000,6,46,11,0
